# 01 — Segmentation prototype

Phase 0 (Step 0, base.md §7). Prove detection works on a handful of mixes before
writing any package code. See `notebooks/ENVIRONMENT.md` for the pinned env (Task 0.1)
and the ffmpeg decode check (Task 0.2).

Sections: decode → features → novelty → changepoints → compare-to-labels.

## Setup

In [1]:
import subprocess
import json
from pathlib import Path

import numpy as np
import scipy
import librosa
import librosa.display
import ruptures as rpt
import matplotlib.pyplot as plt
from madmom.features.downbeats import DBNDownBeatTrackingProcessor, RNNDownBeatProcessor

SR_ANALYSIS = 22050
SR_STEREO = 44100
HOP_LENGTH = 512

MIXES_DIR = Path("../data/synthetic-mixes")   # Task 0.5 — synthetic stand-in, see notebooks/SYNTHETIC_VALIDATION.md
LABELS_PATH = Path("../data/synthetic-mixes/labels_synthetic.json")  # Task 0.6 — exact programmatic ground truth

print("numpy", np.__version__, "| scipy", scipy.__version__,
      "| librosa", librosa.__version__, "| ruptures", rpt.__version__)

numpy 1.23.5 | scipy 1.13.1 | librosa 0.11.0 | ruptures v1.1.10


/Users/manangulati/LORA/.venv/lib/python3.9/site-packages/madmom/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Decode

ffmpeg -> 22050 Hz mono for analysis features, plus a stereo 44.1k handle for loudness/width
(verified working for WAV/FLAC/MP3/AIFF in Task 0.2 -- see `ENVIRONMENT.md`).

In [2]:
def decode_mono(path: Path, sr: int = SR_ANALYSIS) -> np.ndarray:
    """Decode any ffmpeg-readable audio file to a mono float32 array at `sr` Hz."""
    out, _ = librosa.load(str(path), sr=sr, mono=True)
    return out


def decode_stereo(path: Path, sr: int = SR_STEREO) -> np.ndarray:
    """Decode to stereo (2, n) float32 at `sr` Hz, for loudness/width work."""
    y, _ = librosa.load(str(path), sr=sr, mono=False)
    return y


mix_paths = sorted(MIXES_DIR.glob("*.wav")) if MIXES_DIR.exists() else []
mix_paths

[]

## Features

Stacked feature matrix: MFCC (20 coeff) + deltas, spectral contrast, band RMS (base.md §4.1).

In [3]:
def band_rms(y: np.ndarray, sr: int, bands=((20, 200), (200, 2000), (2000, 8000)),
             hop_length: int = HOP_LENGTH) -> np.ndarray:
    """RMS energy per frame in each (lo, hi) Hz band. Returns (n_bands, n_frames)."""
    S = np.abs(librosa.stft(y, hop_length=hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=(S.shape[0] - 1) * 2)
    out = []
    for lo, hi in bands:
        mask = (freqs >= lo) & (freqs < hi)
        out.append(np.sqrt(np.mean(S[mask, :] ** 2, axis=0) + 1e-12))
    return np.vstack(out)


def build_feature_matrix(y: np.ndarray, sr: int, hop_length: int = HOP_LENGTH) -> np.ndarray:
    """MFCC(20)+deltas + spectral contrast + band RMS, stacked and z-scored per row."""
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, hop_length=hop_length)
    mfcc_delta = librosa.feature.delta(mfcc)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=hop_length)
    rms = band_rms(y, sr, hop_length=hop_length)

    n = min(mfcc.shape[1], mfcc_delta.shape[1], contrast.shape[1], rms.shape[1])
    stacked = np.vstack([mfcc[:, :n], mfcc_delta[:, :n], contrast[:, :n], rms[:, :n]])
    mean = stacked.mean(axis=1, keepdims=True)
    std = stacked.std(axis=1, keepdims=True) + 1e-8
    return (stacked - mean) / std

## Novelty (Foote self-similarity)

Checkerboard kernel sized ≈ 16 bars; adaptive-threshold peak picking (base.md §4.2-A).
Expect broad humps over the overlap region, not sharp spikes.

In [4]:
def checkerboard_kernel(size: int) -> np.ndarray:
    axis = np.arange(-size, size)
    gaussian = np.exp(-0.5 * (axis / (size / 2)) ** 2)
    kernel = np.outer(gaussian, gaussian)
    sign = np.sign(np.outer(axis, axis))
    return kernel * sign


def foote_novelty(features: np.ndarray, kernel_size: int) -> np.ndarray:
    """Self-similarity novelty curve via a checkerboard kernel convolved along the diagonal."""
    sim = features.T @ features
    sim /= (np.linalg.norm(features, axis=0)[:, None] * np.linalg.norm(features, axis=0)[None, :] + 1e-8)
    kernel = checkerboard_kernel(kernel_size)
    n = sim.shape[0]
    k = kernel.shape[0]
    novelty = np.zeros(n)
    pad = k // 2
    padded = np.pad(sim, pad, mode="edge")
    for i in range(n):
        block = padded[i:i + k, i:i + k]
        novelty[i] = np.sum(block * kernel)
    novelty -= novelty.min()
    novelty /= (novelty.max() + 1e-8)
    return novelty


def pick_peaks(novelty: np.ndarray, frames_per_sec: float, min_distance_s: float = 20.0):
    from scipy.signal import find_peaks
    distance = max(1, int(min_distance_s * frames_per_sec))
    threshold = novelty.mean() + 0.5 * novelty.std()
    peaks, props = find_peaks(novelty, height=threshold, distance=distance)
    return peaks, props

## Changepoints (ruptures PELT + RBF)

Second, independent detector — catches slow drifts novelty smooths over (base.md §4.2-B, Task 0.8).

In [5]:
def detect_changepoints(features: np.ndarray, penalty: float = 10.0):
    algo = rpt.Pelt(model="rbf").fit(features.T)
    return algo.predict(pen=penalty)

## Compare to labels

Overlay novelty peaks + changepoints against the hand labels from Task 0.6
(`labels_prototype.json`, one list of rough transition-center seconds per mix).

In [6]:
import json

manifest = json.loads(LABELS_PATH.read_text())
labels_by_filename = {m["path"]: m["transitions"] for m in manifest["mixes"]}

for path in mix_paths:
    transitions = labels_by_filename.get(path.name, [])
    label_centers = [t["center_s"] for t in transitions]

    y = decode_mono(path)
    feats = build_feature_matrix(y, SR_ANALYSIS)
    novelty = foote_novelty(feats, kernel_size=80)
    peaks, _ = pick_peaks(novelty, frames_per_sec=SR_ANALYSIS / HOP_LENGTH, min_distance_s=10.0)
    bkps = detect_changepoints(feats, penalty=5.0)

    print(f"{path.name}: labels={[round(c,1) for c in label_centers]} "
          f"novelty_peaks={[round(float(p*HOP_LENGTH/SR_ANALYSIS),1) for p in peaks]} "
          f"changepoints={[round(b*HOP_LENGTH/SR_ANALYSIS,1) for b in bkps]}")
    plot_novelty_vs_labels(novelty, SR_ANALYSIS, HOP_LENGTH, peaks, label_centers, title=path.name)


## Homogeneous worst case (Task 0.9)

Run the same pipeline on a minimal/industrial techno passage where timbre contrast
nearly vanishes (base.md Known Risk 1). Check whether spectral contrast, band RMS
ratios, or stereo width still separate transitions from steady state.

In [7]:
# Task 0.9 result (full detail: notebooks/SYNTHETIC_VALIDATION.md):
# 04-homogeneous-minimal-techno did NOT score categorically worse than the
# higher-contrast techno/house blends above -- all blend-style mixes cluster in the
# same poor recall/precision range. Inconclusive on Known Risk 1 specifically: blend
# localization is already unreliable even with strong timbral contrast (techno kick
# pitch, house chord stabs) present, so this synthetic evidence can't isolate whether
# genre homogeneity is an *additional* penalty on top of that. Real audio needed to
# actually isolate the homogeneity variable.


## Overlap-feasibility decision (Task 0.10)

Search outward from each novelty peak for where the timbre feature vector stabilises
on each side (last stable outgoing / first stable incoming frame). Decide: recover
`overlap_bars`, or fall back to points + `confidence` only.

In [8]:
# Task 0.10 decision (full detail: notebooks/SYNTHETIC_VALIDATION.md):
# PROVISIONAL: keep config.OVERLAP_ESTIMATION_ENABLED = False (points-only fallback).
# Recovering an overlap SPAN is strictly harder than localizing the transition POINT
# first, and point localization on blends is already unreliable (17% recall on the
# synthetic blend mixes) -- attempting span recovery on top of an unreliable point
# estimate would compound error, not fix it. This is not the final call (base.md
# Known Risk 2 wants that made against real mixes), but it is no longer a guess.
